# 02. GraphRAG의 관계 검색 기초: 사람이 만든 작은 그래프

이 노트북은 원문에서 관계를 어떻게 표현하고 조회하는지 살펴보는 단계입니다. `data/graph_facts.json`은 D1–D3를 **사람이 읽고 직접 만든 구조화 결과**이며, 자동 GraphRAG 결과가 아닙니다. 관계 추출의 자동화와 그 오류 가능성은 3번 노트북에서 다룹니다.

| 원문 | 원문에서 확인한 사실 | 기록한 방향 관계 |
| --- | --- | --- |
| D1 | 오헤슬이 결제 시스템 리팩터링을 총괄 | 오헤슬 `-LEADS→` 프로젝트 |
| D2 | 리팩터링 프로젝트가 결제 안정화 프로그램에 속함 | 프로젝트 `-PART_OF→` 프로그램 |
| D3 | 보안팀이 프로그램의 보안 검토를 담당 | 보안팀 `-REVIEWS→` 프로그램 |

사람·프로젝트·프로그램·팀을 노드로, 세 동사를 관계로 정했습니다. 각 관계에는 원문 ID와 근거 문장을 함께 넣었습니다. 첫 셀의 `validate_facts`는 그 근거 문장이 실제 D1–D3에 있는지 확인하지만, 원문에서 관계를 스스로 발견하지는 않습니다.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from session_demo.graph import facts, validate_facts, local_graph, relation_path, evidence_answer, draw_local_graph
import matplotlib.pyplot as plt

model = facts()
validate_facts(model)  # 간선의 근거 문장이 실제 D1–D3 원문에 있는지 확인
display(pd.DataFrame(model["nodes"]))
display(pd.DataFrame(model["edges"]))


## 1. 관계의 이름과 방향 읽기

`local_graph(model)`은 위 노드와 간선을 NetworkX의 **메모리 그래프 자료구조**로 만듭니다. 그림은 그 자료를 보여주는 화면일 뿐, 간선이 그림에서 새로 만들어진 것은 아닙니다. `LEADS`: 오헤슬 → 프로젝트, `PART_OF`: 프로젝트 → 프로그램, `REVIEWS`: 보안팀 → 프로그램입니다. 프로그램에 두 대상이 연결되어 있어도 **오헤슬 ↔ 보안팀 직접 협업** 간선은 없습니다.


In [ ]:
graph = local_graph(model)
fig = draw_local_graph(graph, ROOT / "runs" / "figures" / "small_graph.png")
display(fig)
plt.close(fig)
print("그림 저장:", ROOT / "runs" / "figures" / "small_graph.png")


## 2. 경로와 세 출처를 함께 회수

질문: **“오헤슬과 보안팀은 어떤 관계인가?”** 그래프에는 D1·D2·D3의 **개별 간선**이 저장돼 있습니다. 두 사람·팀의 연결을 설명하는 경로가 별도 정답으로 저장된 것은 아닙니다. `relation_path`는 오헤슬에서 시작해 `LEADS → PART_OF ← REVIEWS`라는 관계 종류와 방향을 만족하는 간선을 차례로 확인하고, 세 관계와 출처를 돌려줍니다. 무방향 최단 경로 검색은 아닙니다.

노드가 네 개인 이 예제에서는 경로를 눈으로도 알 수 있습니다. 이 조회는 새로운 사실을 발견하거나 임의의 자연어 질문을 처리하기보다 **관계 패턴을 조회하는 방식**을 보여줍니다. LLM은 호출하지 않습니다. `evidence_answer`는 경로가 확인되면 미리 작성한 문장을, 경로가 없으면 근거가 부족하다는 문장을 반환합니다.


In [ ]:
path = relation_path(graph)
display(pd.DataFrame([{
    "시작": graph.nodes[edge["from"]]["name"],
    "관계": edge["kind"],
    "도착": graph.nodes[edge["to"]]["name"],
    "출처": edge["source_id"],
    "원문 근거": edge["evidence"],
} for edge in path]))
print("[규칙 기반 예시 문장 · LLM 호출 없음]")
print(evidence_answer(path))


## 3. 관계가 하나 빠지면 경로도 끊긴다

그래프도 누락된 간선을 자동으로 복원하지 않습니다. D2 관계를 제거한 복사본에서 같은 패턴을 찾아 봅니다. 원래 그래프는 그대로 둡니다.


In [ ]:
import copy
incomplete = copy.deepcopy(model)
incomplete["edges"] = [edge for edge in incomplete["edges"] if edge["source_id"] != "D2"]
broken_graph = local_graph(incomplete)
broken_path = relation_path(broken_graph)
print("D2가 빠진 경로:", broken_path)
print(evidence_answer(broken_path))


## 4. Neo4j Browser에서 확인하기 (선택)

앞의 1~3번은 `graph_facts.json`에 사람이 확인해 적어 둔 관계를 **NetworkX 메모리 그래프**로 읽은 것입니다. 4번도 **바로 그 동일한 자료**를 사용합니다. `write_neo4j(driver, model)`이 네 노드와 세 관계를 Neo4j에 `MERGE`하고, Cypher로 같은 경로와 출처를 조회합니다. Neo4j가 D1–D3 원문을 분석해 관계를 자동 추출하는 과정은 없습니다. NetworkX도 앞에서 실제 그래프 구조를 만들어 탐색했습니다. Neo4j는 같은 구조를 **프로세스 밖의 데이터베이스에 보관**하고 Cypher로 조회하며 Browser에서 살펴볼 수 있게 합니다. 이 네 노드 예제만으로 Neo4j가 더 정확하거나 꼭 필요하다고 결론 내릴 수는 없습니다.

Neo4j 인스턴스를 켜고 `.env`에 접속 정보를 설정한 뒤 `CONNECT_NEO4J`를 `True`로 바꿉니다. 다른 노드는 삭제하지 않습니다. 출력된 Cypher를 Neo4j Browser에 붙여 넣으면 저장된 관계를 그래프로 볼 수 있습니다.


In [ ]:
CONNECT_NEO4J = False
if CONNECT_NEO4J:
    from session_demo.graph import neo4j_driver, write_neo4j, query_neo4j_path, VISUAL_QUERY
    with neo4j_driver() as driver:
        write_neo4j(driver, model)
        display(pd.DataFrame(query_neo4j_path(driver)))
    print("Neo4j Browser 그래프 시각화용 Cypher:\n", VISUAL_QUERY)
else:
    print("Neo4j 연결을 건너뛰었습니다. 위 로컬 그림과 경로는 이미 실행되었습니다.")


**무엇을 확인했나:** 사람이 만든 그래프에서 관계의 종류·방향·출처를 확인하고, 질문에 필요한 경로를 조회했습니다. 간선이 빠지면 경로도 끊깁니다. Neo4j는 같은 데이터를 저장·조회하는 다른 방식이며, 이 예제는 GraphRAG의 자동 추출 성능이나 답변 품질 우위를 증명하지 않습니다. `03_microsoft_graphrag.ipynb`에서는 원문을 자동 인덱싱했을 때 실제로 어떤 관계와 산출물이 나오는지 확인합니다.
